# Interaction Modeling

This notebook defines and constructs the behavioral interaction model for the Retailrocket recommendation dataset.

The recommendation system uses implicit feedback from user behavior.

In [1]:
from pathlib import Path
import pandas as pd

# Locate the project root reliably from the notebook working directory.
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent:
    if (PROJECT_ROOT / "data" / "raw").exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

EVENTS_PATH = RAW_DIR / "events.csv"
OUTPUT_PATH = PROCESSED_DIR / "interaction_records.csv"

print("Project root:", PROJECT_ROOT)
print("Events:", EVENTS_PATH)
print("Output:", OUTPUT_PATH)

Project root: f:\annuspeaks.com\recommendation-system
Events: f:\annuspeaks.com\recommendation-system\data\raw\events.csv
Output: f:\annuspeaks.com\recommendation-system\data\processed\interaction_records.csv


In [2]:
INTERACTION_WEIGHTS = {
    "view": 1,
    "addtocart": 3,
    "transaction": 5,
}

INTERACTION_TYPES = {
    "view": "Product viewed",
    "addtocart": "Product added to cart",
    "transaction": "Product purchased",
}

print("Interaction types:")
for event, meaning in INTERACTION_TYPES.items():
    print(f"{event:12} -> {meaning}")

print("\nInteraction weights:")
for event, weight in INTERACTION_WEIGHTS.items():
    print(f"{event:12} -> {weight}")

Interaction types:
view         -> Product viewed
addtocart    -> Product added to cart
transaction  -> Product purchased

Interaction weights:
view         -> 1
addtocart    -> 3
transaction  -> 5


## Interaction Model

| Event | Weight | Interpretation |
|---|---:|---|
| `view` | 1 | Weak positive interest |
| `addtocart` | 3 | Stronger purchase intent |
| `transaction` | 5 | Strongest positive signal |

The system uses implicit behavioral feedback rather than explicit ratings.

In [3]:
# Construct user-item interaction records.
# Records are written incrementally so the complete event dataset is not kept in RAM.

if OUTPUT_PATH.exists():
    OUTPUT_PATH.unlink()

total_records = 0
event_counts = {}

usecols = [
    "timestamp",
    "visitorid",
    "event",
    "itemid",
    "transactionid",
]

for chunk in pd.read_csv(
    EVENTS_PATH,
    usecols=usecols,
    chunksize=250_000,
):
    chunk = chunk[chunk["event"].isin(INTERACTION_WEIGHTS)].copy()
    chunk["weight"] = chunk["event"].map(INTERACTION_WEIGHTS)

    chunk = chunk.rename(columns={
        "visitorid": "user_id",
        "itemid": "item_id",
        "event": "interaction_type",
    })

    chunk = chunk[[
        "user_id",
        "item_id",
        "interaction_type",
        "weight",
        "timestamp",
        "transactionid",
    ]]

    chunk.to_csv(
        OUTPUT_PATH,
        mode="a",
        header=not OUTPUT_PATH.exists(),
        index=False,
    )

    total_records += len(chunk)
    for event, count in chunk["interaction_type"].value_counts().items():
        event_counts[event] = event_counts.get(event, 0) + int(count)

print("Interaction records created:", f"{total_records:,}")
print("\nInteraction counts:")
print(pd.Series(event_counts).sort_index())
print("\nOutput:")
print(OUTPUT_PATH)

Interaction records created: 2,756,101

Interaction counts:
addtocart        69332
transaction      22457
view           2664312
dtype: int64

Output:
f:\annuspeaks.com\recommendation-system\data\processed\interaction_records.csv


In [4]:
# Verify the generated interaction dataset without loading the whole file.

sample = pd.read_csv(OUTPUT_PATH, nrows=10)
print("Columns:")
print(sample.columns.tolist())
            
print("\nSample:")
display(sample)
print("\nFile size:", f"{OUTPUT_PATH.stat().st_size / (1024**2):.2f} MB")

Columns:
['user_id', 'item_id', 'interaction_type', 'weight', 'timestamp', 'transactionid']

Sample:


,user_id,item_id,interaction_type,weight,timestamp,transactionid
0,257597,355908,view,1,1433221332117,NaN
1,992329,248676,view,1,1433224214164,NaN
2,111016,318965,view,1,1433221999827,NaN
3,483717,253185,view,1,1433221955914,NaN
4,951259,367447,view,1,1433221337106,NaN
5,972639,22556,view,1,1433224086234,NaN
6,810725,443030,view,1,1433221923240,NaN
7,794181,439202,view,1,1433223291897,NaN
8,824915,428805,view,1,1433220899221,NaN
9,339335,82389,view,1,1433221204592,NaN



File size: 97.80 MB


## Ratings / Reviews

The verified Retailrocket interaction schema does not contain explicit user ratings or review scores.

Therefore:

- Ratings do not contribute to the current preference signal.
- Reviews do not contribute to the current preference signal.
- Preference is derived from behavioral interactions.

## Negative / Unknown Interactions

The dataset does not provide explicit negative-feedback events.

Therefore:

- `view`, `addtocart`, and `transaction` are treated as positive behavioral signals with different strengths.
- A user-item pair with no observed interaction is treated as unknown, not negative.
- Non-interaction will not be randomly labeled as negative.
- Negative sampling, if required by a later recommendation model, will be defined during model training.

In [5]:
# Final Phase 2.1 validation

total = 0
users = set()
items = set()
types = {}
weights = {}
missing = None

for chunk in pd.read_csv(OUTPUT_PATH, chunksize=250_000):
    total += len(chunk)
    users.update(chunk["user_id"].dropna().unique())
    items.update(chunk["item_id"].dropna().unique())

    for value, count in chunk["interaction_type"].value_counts().items():
        types[value] = types.get(value, 0) + int(count)

    for value, count in chunk["weight"].value_counts().items():
        weights[value] = weights.get(value, 0) + int(count)

    current_missing = chunk.isna().sum()
    missing = current_missing if missing is None else missing.add(current_missing, fill_value=0)

print("Total interaction records:", f"{total:,}")
print("Unique users:", f"{len(users):,}")
print("Unique products:", f"{len(items):,}")
            
print("\nInteraction types:")
print(pd.Series(types).sort_index())
print("\nWeights:")
print(pd.Series(weights).sort_index())
print("\nMissing values:")
print(missing.astype(int))

Total interaction records: 2,756,101
Unique users: 1,407,580
Unique products: 235,061

Interaction types:
addtocart        69332
transaction      22457
view           2664312
dtype: int64

Weights:
1    2664312
3      69332
5      22457
dtype: int64

Missing values:
user_id                   0
item_id                   0
interaction_type          0
weight                    0
timestamp                 0
transactionid       2733644
dtype: int64


## Completion Criteria

- Interaction types are defined.
- Interaction weights are defined.
- User-item interaction records are constructed in `data/processed/interaction_records.csv`.
- Ratings/reviews are explicitly excluded because no verified explicit rating/review signal is available.
- Non-interactions are treated as unknown rather than negative.
